In [3]:
from pathlib import Path

PACKAGE = Path("/kaggle/input/datasets/adeelumar17/bitext")

print("Package files:")
for path in sorted(PACKAGE.iterdir()):
    print(path.name)

assert (PACKAGE / "adapter").exists()
assert (PACKAGE / "tokenizer").exists()
assert (PACKAGE / "generation_config.json").exists()
assert (PACKAGE / "profile.json").exists()
assert (PACKAGE / "core.py").exists()
assert (PACKAGE / "model_runtime.py").exists()

Package files:
README_DIAGNOSTIC_ONLY.md
adapter
core.py
environment.json
environment.lock.txt
failure_report.json
generation_config.json
length_report.json
load_generator.py
metrics.py
model_runtime.py
preparation_report.json
prepare.py
profile.json
run.py
tokenizer
train.py
worker.py


In [4]:
# Install only the inference/runtime dependencies.
import sys, subprocess

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "unsloth==2026.10.10",
    "unsloth_zoo==2026.10.10",
    "transformers==5.17.0",
    "peft==0.21.2",
    "bitsandbytes==0.50.2",
    "sentencepiece>=0.2,<0.3",
    "huggingface_hub>=1.5.0,<2.0.0",
], check=True)

print("Runtime installed.")

Runtime installed.


In [5]:
# Load the recovered validation-selected adapter.
import os, sys, json, torch

assert torch.cuda.is_available(), "Enable a GPU in Kaggle settings."
sys.path.insert(0, str(PACKAGE))
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"

from core import read_json, canonical
from model_runtime import Generator, exported_records

config = read_json(PACKAGE / "generation_config.json")
profile = read_json(PACKAGE / "profile.json")

generator = Generator(
    config,
    profile,
    adapter=PACKAGE / "adapter",
    temperature=config["temperature"],
)

print("Loaded:", config["model_id"])
print("Selected temperature:", config["temperature"])
print("Validation acceptance passed:", config["validation_acceptance_passed"])

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

==((====))==  Unsloth 2026.10.10: Fast Gemma4 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma4 won't work! Using float32.


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

Unsloth: the model has bfloat16 layers on GPU 0 (Tesla T4), which has no native bfloat16, so they run slowly. Load with dtype = torch.float16, or keep the model on bfloat16 GPUs with CUDA_VISIBLE_DEVICES.
Loaded: google/gemma-4-E4B-it
Selected temperature: 1.0
Validation acceptance passed: True


In [6]:
# Generate a small validated sample first.
records = generator.generate_synthetic(
    n=20,
    seed=42,
    mode="validated",
)
records = exported_records(records, profile["kind"], seed=42)

print(json.dumps(generator.last_report, indent=2))
print(json.dumps(records[:3], indent=2, ensure_ascii=False))

{
  "requested": 20,
  "requests_processed": 20,
  "attempts": 20,
  "accepted": 20,
  "returned": 20,
  "shortfall": 0,
  "rejections": {},
  "complete": true,
  "mode": "validated"
}
[
  {
    "category": "HOME_INSURANCE",
    "instruction": "I want to know about my fucking home insurance, find some information",
    "intent": "information_home_insurance",
    "response": "To access comprehensive information regarding your home insurance policy, please adhere to the following instructions:\n\n1. Go to {{WEBSITE_URL}}.\n2. Sign in to your account with your username and password.\n3. Click on the {{HOME_INSURANCE_SECTION}} tab available in the navigation menu.\n4. Choose your policy number to obtain detailed insights.\n\nShould you require additional assistance, please do not hesitate to reach out to our support team via the {{HELP_SECTION}} located on {{WEBSITE_URL}}.",
    "tags": "BCW"
  },
  {
    "category": "AUTO_INSURANCE",
    "instruction": "I would like information about my a

In [7]:
# Save the generated synthetic records.
output = Path("/kaggle/working/bitext_synthetic_validated_20.jsonl")
with output.open("w", encoding="utf-8") as f:
    for record in records:
        f.write(canonical(record) + "\n")

print(output)

/kaggle/working/bitext_synthetic_validated_20.jsonl
